# 接触对象：补充接触网络

度分布、最大接触簇和效率仅作补充，不作为折叠度。全部残基为节点，主链不计边。
主要分析请看接触数、序列概率及Rg。分布阴影为块SD，均值比较误差为块SEM。

在首个代码单元选择OBJECT：central_chain为每帧中心链，dense_chains为全部浓相链。


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "simulations.yaml").is_file())
sys.path.insert(0, str(ROOT))
from common import inputs
from common import results as files

SELECTED = None
OBJECT = "central_chain"  # central_chain / dense_chains
RESULTS_ROOT = ROOT / "03_results"
SIMULATIONS = inputs.select_simulations(ROOT / "simulations.yaml", SELECTED)
RESULTS = {s["id"]: files.completed(RESULTS_ROOT / s["id"] / "02_contacts/01_intra" / OBJECT) for s in SIMULATIONS}
SOURCES = {name: files.read_json(path / "metadata.json") for name, path in RESULTS.items()}
COLORS = {name: plt.get_cmap("tab10")(i % 10) for i, name in enumerate(RESULTS)}
FIGURES = ROOT / "05_plot/02_contacts/01_intra/figures" / OBJECT
FIGURES.mkdir(parents=True, exist_ok=True)
plt.rcParams.update(
    {
        "font.size": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "svg.fonttype": "none",
        "pdf.fonttype": 42,
        "legend.frameon": False,
    }
)


def export(fig, name):
    for key, path in RESULTS.items():
        files.completed(path)
        if files.read_json(path / "metadata.json") != SOURCES[key]:
            raise ValueError("Results changed; restart notebook")
    for ext in ("png", "pdf", "svg"):
        fig.savefig(FIGURES / f"{name}.{ext}", dpi=300, bbox_inches="tight")
    files.write_json(FIGURES / f"{name}.sources.json", SOURCES)


In [ ]:
summary = pd.concat([pd.read_csv(p / "summary.csv") for p in RESULTS.values()], ignore_index=True)


def mean_comparison(metrics, filename):
    fig, axes = plt.subplots(
        1, len(metrics), figsize=(4.5 * len(metrics), 3.5), squeeze=False, constrained_layout=True
    )
    for ax, (metric, label) in zip(axes[0], metrics.items()):
        bounds = []
        for x, name in enumerate(RESULTS):
            row = summary[summary.simulation_id.eq(name)].iloc[0]
            value, sem = row[metric + "_mean_frames"], row[metric + "_sem_blocks"]
            if np.isfinite(value):
                err = sem if np.isfinite(sem) else 0
                bounds.extend((value - err, value + err))
                ax.errorbar(x, value, yerr=err, fmt="o", capsize=4, color=COLORS[name])
        if bounds:
            pad = max((max(bounds) - min(bounds)) * 0.15, abs(max(bounds)) * 0.02, 1e-9)
            ax.set_ylim(max(0, min(bounds) - pad), max(bounds) + pad)
        ax.set(xticks=range(len(RESULTS)), xticklabels=list(RESULTS), ylabel=label)
        ax.tick_params(axis="x", labelrotation=20)
    fig.suptitle("Frame-equal mean ± block SEM", fontsize=10)
    export(fig, filename)
    plt.show()


mean_comparison(
    {
        "largest_component_fraction": "Largest contact cluster / residues",
        "global_efficiency": "Contact-network efficiency",
    },
    "supplementary_network",
)


In [ ]:
DEGREE_MAX = 8
fig, ax = plt.subplots(figsize=(6, 3.5), constrained_layout=True)
for name, path in RESULTS.items():
    data = pd.read_csv(path / "degree_distribution/summary.csv").sort_values("degree")
    x, y, sd = (
        data.degree.to_numpy(),
        data.fraction_mean_frames.to_numpy(),
        data.fraction_std_blocks.to_numpy(),
    )
    ax.plot(x, y, marker="o", ms=3, color=COLORS[name], label=name)
    ax.fill_between(x, np.maximum(0, y - sd), np.minimum(1, y + sd), color=COLORS[name], alpha=0.15)
ax.set(
    xlim=(0, DEGREE_MAX),
    xticks=range(DEGREE_MAX + 1),
    xlabel="Residue contact degree k",
    ylabel="P(k)",
    ylim=(0, None),
)
ax.legend()
export(fig, "supplementary_degree_distribution")
plt.show()
